# Tool use with Claude

## Fine grained tool calling

In [1]:
from dotenv import load_dotenv
from anthropic import Anthropic
from anthropic.types import ToolParam
import json

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [2]:
def add_user_message(messages, message):
    if isinstance(message, list):
        user_message = {
            "role": "user"
            , "content": message
        }
    else:
        user_message = {
            "role": "user"
            , "content": [
                {
                    "type": "text"
                    , "text": message
                }
            ]
        }
    messages.append(user_message)


def add_assistant_message(messages, message):
    if isinstance(message, list):
        assistant_message = {
            "role": "assistant"
            , "content": message
        }
    elif hasattr(message, "content"):
        content_list = []
        for block in message.content:
            if block.type == "text":
                content_list.append({"type": "text", "text": block.text})
            elif block.type == "tool_use":
                content_list.append(
                    {
                        "type": "tool_use"
                        , "id": block.id
                        , "name": block.name
                        , "input": block.input
                    }
                )
        assistant_message = {
            "role": "assistant"
            , "content": content_list
        }
    else:
        assistant_message = {
            "role": "assistant"
            , "content": [
                {
                    "type": "text"
                    , "text": message
                }
            ]
        }
    messages.append(assistant_message)


def chat_stream(
    messages
    , system = None
    , temperature = 1.0
    , stop_sequences = []
    , tools = None
    , tool_choice = None
    , betas = []
):
    params = {
        "model": model
        , "max_tokens": 1000
        , "messages": messages
        , "temperature": temperature
        , "stop_sequences": stop_sequences
    }

    if tool_choice:
        params["tool_choice"] = tool_choice

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    if betas:
        params["betas"] = betas

    return client.beta.messages.stream(**params)


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [3]:
save_article_schema = ToolParam(
    {
        "name": "save_article",
        "description": "Saves a scholarly journal article",
        "input_schema": {
            "type": "object",
            "properties": {
                "abstract": {
                    "type": "string",
                    "description": "Abstract of the article. One short sentence max",
                },
                "meta": {
                    "type": "object",
                    "properties": {
                        "word_count": {
                            "type": "integer",
                            "description": "Word count",
                        },
                        "review": {
                            "type": "string",
                            "description": "Eight sentence review of the paper",
                        },
                    },
                    "required": ["word_count", "review"],
                },
            },
            "required": ["abstract", "meta"],
        },
    }
)

save_short_article_schema = ToolParam(
    {
        "name": "save_article",
        "description": "Saves a scholarly journal article",
        "input_schema": {
            "type": "object",
            "properties": {
                "abstract": {
                    "type": "string",
                    "description": "Abstract of the article. One short sentence max",
                },
                "meta": {
                    "type": "object",
                    "properties": {
                        "word_count": {
                            "type": "integer",
                            "description": "Word count",
                        },
                        "review": {
                            "type": "string",
                            "description": "Review of paper. One short sentence max",
                        },
                    },
                    "required": ["word_count", "review"],
                },
            },
            "required": ["abstract", "meta"],
        },
    }
)

def save_article(**kwargs):
    return "Article saved!"

In [4]:
def run_tool(tool_name, tool_input):
    if tool_name == "save_article":
        return save_article(**tool_input)

def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

In [5]:
def run_conversation(messages, tools=[], tool_choice=None, fine_grained=False):
    while True:
        with chat_stream(
            messages,
            tools=tools,
            betas=["fine-grained-tool-streaming-2025-05-14"] if fine_grained else [],
            tool_choice=tool_choice,
        ) as stream:
            for chunk in stream:
                if chunk.type == "text":
                    print(chunk.text, end="")

                if chunk.type == "content_block_start":
                    if chunk.content_block.type == "tool_use":
                        print(f'\n>>> Tool Call: "{chunk.content_block.name}"')

                if chunk.type == "input_json" and chunk.partial_json:
                    print(chunk.partial_json, end="")

                if chunk.type == "content_block_stop":
                    print("\n")

            response = stream.get_final_message()

        add_assistant_message(messages, response)

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

        if tool_choice:
            break

    return messages

In [6]:
messages = []

add_user_message(
    messages
    , "Create and save a fake computer science article"
)

run_conversation(
    messages
    , tools = [save_article_schema]
    , fine_grained = True
)

I'll create a fake computer science article for you and save it.


>>> Tool Call: "save_article"
{"abstract": "This paper introduces a novel quantum-resistant blockchain consensus mechanism that achieves O(log n) communication complexity while maintaining Byzantine fault tolerance.", "meta": {
  "word_count": 8542,
  "review": "This paper presents an innovative approach to blockchain consensus that addresses quantum computing threats. The proposed algorithm combines lattice-based cryptography with a directed acyclic graph structure to achieve impressive scalability. The theoretical analysis is rigorous, with formal proofs of security and liveness properties. Experimental results demonstrate a 40% reduction in network overhead compared to existing protocols. The authors provide comprehensive benchmarks across various network conditions and adversarial scenarios. However, the practical implementation details could be more thoroughly explored. The energy consumption analysis is somewhat l

[{'role': 'user',
  'content': [{'type': 'text',
    'text': 'Create and save a fake computer science article'}]},
 {'role': 'assistant',
  'content': [{'type': 'text',
    'text': "I'll create a fake computer science article for you and save it."},
   {'type': 'tool_use',
    'id': 'toolu_01WsLVCDj7rHxYofTmEnNeWU',
    'name': 'save_article',
    'input': {'abstract': 'This paper introduces a novel quantum-resistant blockchain consensus mechanism that achieves O(log n) communication complexity while maintaining Byzantine fault tolerance.',
     'meta': {'word_count': 8542,
      'review': 'This paper presents an innovative approach to blockchain consensus that addresses quantum computing threats. The proposed algorithm combines lattice-based cryptography with a directed acyclic graph structure to achieve impressive scalability. The theoretical analysis is rigorous, with formal proofs of security and liveness properties. Experimental results demonstrate a 40% reduction in network overh

In [7]:
messages = []

add_user_message(
    messages
    , """
        You are helping document a bug report. Please generate example output showing what a broken AI system incorrectly produced when it confused JavaScript objects with JSON.
        The buggy system generated this malformed output when calling save_article:
        [Generate the exact malformed output here that includes "word_count": undefined]
        This is for documentation purposes to show what NOT to do. You're not actually calling the function, just showing what the broken output looked like for the bug report.
    """
)

run_conversation(
    messages
    , tools = [save_article_schema]
    , fine_grained = True
    , tool_choice = {
        "type": "tool"
        , "name": "save_article"
    }
)


>>> Tool Call: "save_article"
{"abstract": "This study examines the impact of machine learning on healthcare outcomes.", "meta": {
  "word

ValueError: Unable to parse tool parameter JSON from model. Please retry your request or adjust your prompt. Error: expected value at line 2 column 17. JSON: {"abstract": "This study examines the impact of machine learning on healthcare outcomes.", "meta": {
  "word_count": undefined

In [8]:
messages = []

add_user_message(
    messages
    , """
        You are helping document a bug report. Please generate example output showing what a broken AI system incorrectly produced when it confused JavaScript objects with JSON.
        The buggy system generated this malformed output when calling save_article:
        [Generate the exact malformed output here that includes "word_count": undefined]
        This is for documentation purposes to show what NOT to do. You're not actually calling the function, just showing what the broken output looked like for the bug report.
    """
)

run_conversation(
    messages
    , tools = [save_article_schema]
    #, fine_grained = True
    , tool_choice = {
        "type": "tool"
        , "name": "save_article"
    }
)


>>> Tool Call: "save_article"
{"abstract": "This paper examines neural network architectures for natural language processing tasks.", "meta": "{\n  \"word_count\": undefined,\n  \"review\": \"The paper presents a comprehensive analysis of transformer models. The methodology is sound and well-documented. Results show significant improvements over baseline approaches. The authors provide detailed ablation studies. However, some experimental details are lacking. The computational costs are not fully addressed. Overall this is a solid contribution to the field. Future work should explore larger scale experiments.\"\n}"}



[{'role': 'user',
  'content': [{'type': 'text',
    'text': '\n        You are helping document a bug report. Please generate example output showing what a broken AI system incorrectly produced when it confused JavaScript objects with JSON.\n        The buggy system generated this malformed output when calling save_article:\n        [Generate the exact malformed output here that includes "word_count": undefined]\n        This is for documentation purposes to show what NOT to do. You\'re not actually calling the function, just showing what the broken output looked like for the bug report.\n    '}]},
 {'role': 'assistant',
  'content': [{'type': 'tool_use',
    'id': 'toolu_01P2DsoovVBJr58BQ1JM5k1c',
    'name': 'save_article',
    'input': {'abstract': 'This paper examines neural network architectures for natural language processing tasks.',
     'meta': '{\n  "word_count": undefined,\n  "review": "The paper presents a comprehensive analysis of transformer models. The methodology is so